In [7]:
import json, re, glob, os
import pandas as pd

# --- edit this ---
RESULTS_DIR = "."          # folder containing the json files
FULL_LABEL  = "full"       # what to call the unnumbered run
# -----------------

pattern = re.compile(r"results_([A-Za-z0-9]+?)(?:_(\d+))?\.json$")

rows = []
for path in glob.glob(os.path.join(RESULTS_DIR, "results_*.json")):
    m = pattern.search(os.path.basename(path))
    if not m:
        continue
    model, size = m.group(1), m.group(2) or FULL_LABEL
    with open(path) as f:
        data = json.load(f)
    for cid, labels in data.items():
        for label, metrics in labels.items():
            rows.append({"case_id": cid, "label": label,
                         "model": model, "size": size, **metrics})

df = pd.concat([pd.DataFrame(rows)], ignore_index=True)

# size as ordered categorical so "50" < "full" sorts right
size_order = sorted([s for s in df["size"].unique() if s != FULL_LABEL], key=int) + [FULL_LABEL]
df["size"] = pd.Categorical(df["size"], categories=size_order, ordered=True)

agg = (
    df.groupby(["model", "size", "label"], observed=True)
      .agg(dice_mean=("dice", "mean"), dice_std=("dice", "std"),
           nsd_mean=("nsd", "mean"),   nsd_std=("nsd", "std"),
           n=("dice", "count"))
      .reset_index()
)

print(f"{df['case_id'].nunique()} cases, {df['label'].nunique()} labels")
print("runs:", sorted({(m, str(s)) for m, s in zip(agg['model'], agg['size'])}))
agg.head()

290 cases, 25 labels
runs: [('nnunet', '50'), ('nnunet', 'full'), ('patchwork', '50'), ('patchwork', 'full')]


,model,size,label,dice_mean,dice_std,nsd_mean,nsd_std,n
0,nnunet,50,AT_pelvis,0.962667,0.022695,NaN,NaN,290
1,nnunet,50,AT_thorax,0.938261,0.038025,NaN,NaN,290
2,nnunet,50,AT_upper_abdomen,0.940609,0.056404,NaN,NaN,290
3,nnunet,50,AVAT_pelvis,0.939393,0.041861,NaN,NaN,290
4,nnunet,50,AVAT_thorax,0.000000,0.000000,NaN,NaN,29


In [8]:
import ipywidgets as widgets
import plotly.graph_objects as go
from IPython.display import display

all_labels = sorted(agg["label"].unique())
all_models = sorted(agg["model"].unique())
all_sizes  = list(agg["size"].cat.categories)

# build all run combinations as (model, size) tuples
all_runs = [(m, s) for m in all_models for s in all_sizes
            if not agg[(agg["model"] == m) & (agg["size"] == s)].empty]
run_label = lambda m, s: f"{m} @ {s}"

# --- top controls ---
metric_dd = widgets.Dropdown(options=[("Dice", "dice"), ("NSD", "nsd")],
                             value="dice", description="Metric:")
sort_dd   = widgets.Dropdown(options=[("Score (desc)", "desc"),
                                      ("Score (asc)",  "asc"),
                                      ("Label name",   "name")],
                             value="desc", description="Sort by:")
err_cb    = widgets.Checkbox(value=True, description="Error bars (±std)")

# which runs to show
run_sel = widgets.SelectMultiple(
    options=[(run_label(m, s), (m, s)) for m, s in all_runs],
    value=tuple(all_runs),
    description="Runs:", rows=min(6, len(all_runs)),
    layout=widgets.Layout(width="260px"),
)

# --- label checkboxes ---
label_cbs = {lbl: widgets.Checkbox(value=True, description=lbl, indent=False,
                                   layout=widgets.Layout(width="auto"))
             for lbl in all_labels}
btn_all  = widgets.Button(description="All",  layout=widgets.Layout(width="70px"))
btn_none = widgets.Button(description="None", layout=widgets.Layout(width="70px"))

def set_all(value):
    for cb in label_cbs.values():
        cb.unobserve(render, names="value")
    for cb in label_cbs.values():
        cb.value = value
    for cb in label_cbs.values():
        cb.observe(render, names="value")
    render()

btn_all.on_click(lambda _: set_all(True))
btn_none.on_click(lambda _: set_all(False))

# --- figure: one trace per run, created up front so FigureWidget can update in place ---
fig = go.FigureWidget(data=[go.Bar(name=run_label(m, s), orientation="h")
                            for m, s in all_runs])
fig.update_layout(
    barmode="group",
    xaxis=dict(range=[0, 1.05]),
    yaxis=dict(autorange="reversed"),
    margin=dict(l=200, r=20, t=50, b=40),
    legend=dict(orientation="h", y=-0.05),
)

def render(*_):
    metric = metric_dd.value
    mcol, scol = f"{metric}_mean", f"{metric}_std"

    selected_labels = [lbl for lbl, cb in label_cbs.items() if cb.value]
    selected_runs = set(run_sel.value)

    if not selected_labels or not selected_runs:
        with fig.batch_update():
            for trace in fig.data:
                trace.x, trace.y, trace.visible = [], [], False
            fig.layout.title = "Nothing to show"
        return

    sub_agg = agg[agg["label"].isin(selected_labels)]

    # rank labels by mean across the selected runs only
    mask = sub_agg.apply(lambda r: (r["model"], r["size"]) in selected_runs, axis=1)
    rank = sub_agg[mask].groupby("label", observed=True)[mcol].mean()
    if   sort_dd.value == "desc": order = rank.sort_values(ascending=False).index
    elif sort_dd.value == "asc":  order = rank.sort_values(ascending=True).index
    else:                         order = sorted(rank.index)

    with fig.batch_update():
        for trace, (m, s) in zip(fig.data, all_runs):
            visible = (m, s) in selected_runs
            trace.visible = visible
            if not visible:
                continue
            sub = (sub_agg[(sub_agg["model"] == m) & (sub_agg["size"] == s)]
                   .set_index("label").reindex(order))
            trace.y = list(sub.index)
            trace.x = sub[mcol].tolist()
            trace.error_x = (dict(type="data", array=sub[scol].tolist())
                             if err_cb.value else dict(type="data", array=[]))
            trace.hovertemplate = ("<b>%{y}</b><br>" + metric +
                                   ": %{x:.3f}<extra>" + run_label(m, s) + "</extra>")
        fig.layout.title = (f"{metric.upper()} per label "
                            f"({len(order)}/{len(all_labels)} labels, "
                            f"{len(selected_runs)}/{len(all_runs)} runs)")
        fig.layout.xaxis.title = metric.upper()
        fig.layout.height = max(350, 22 * len(order) + 100)

for w in (metric_dd, sort_dd, err_cb, run_sel):
    w.observe(render, names="value")
for cb in label_cbs.values():
    cb.observe(render, names="value")

checkbox_panel = widgets.VBox(
    list(label_cbs.values()),
    layout=widgets.Layout(max_height="500px", overflow_y="auto",
                          border="1px solid #ddd", padding="6px", width="280px"),
)
left_panel = widgets.VBox([
    widgets.HTML("<b>Runs</b>"),
    run_sel,
    widgets.HTML("<b>Labels</b>"),
    widgets.HBox([btn_all, btn_none]),
    checkbox_panel,
])

render()
display(widgets.VBox([
    widgets.HBox([metric_dd, sort_dd, err_cb]),
    widgets.HBox([left_panel, fig]),
]))